# Mini Project
##Airline Tweet Sentiment Classifier using NLP

*Notes:
Use sample datasets from https://github.com/salman1256/aimltraining/blob/main/Day-30/airline_tweets_sample.csv*

**Steps**
1. Import libraries
2. Load and explore datasets
3. Clean and preprocess the text
4. Convert text to numerical vectors (TF-IDF)
5. Split into train and test sets
6. Train a Logistic Regression model
7. Evaluate accuracy and classification report
8. Predict sentiment for new example tweets

In [88]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [10]:
# Step 1b: Download nltk required thing like stopwords
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [89]:
# Step 2a: Load datasets
url = "https://raw.githubusercontent.com/salman1256/aimltraining/main/Day-30/airline_tweets_sample.csv"
df = pd.read_csv(url)

In [90]:
# Step 2b: Checks datasets at least top 5 values
print("Top 5 Rows")
df.head()

Top 5 Rows


,text,sentiment
0,"@United flight was delayed for 3 hours, worst ...",negative
1,"Loved the service on @Delta, crew was super fr...",positive
2,"@AmericanAir lost my luggage again, so disappo...",negative
3,Smooth boarding and on-time arrival. Great job...,positive
4,The seats were uncomfortable but staff was polite,neutral


In [91]:
# Step 3: Text Cleaning and Preprocessing
# For each tweet do:
# a. Convert to lowercase
# b. Remove URLs
# c. Remove special characters and numbers
# d. Remove stopwords (common words like *the, is, and* etc.)
# e. Apply ** stemming ** (reduce words to their root form)

stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text_with_stemming(text):
    # Convert to lowercase, Remove URLs, Remove special characters and numbers
    # Remove mentions/URLs first
    text = re.sub(r'@\w+|http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Convert to lowercase and remove non-letters
    text = re.sub(r'[^a-zA-Z\s]', ' ', text.lower())

    # Tokenize
    tokens = text.split()
    processed_tokens = []

    for word in tokens:
        # Remove stopwords
        if word not in stop_words:
            # Apply stemming
            processed_tokens.append(stemmer.stem(word))

    return " ".join(processed_tokens)

df['cleaned_text'] = df['text'].apply(preprocess_text_with_stemming)
print("Sample Preprocessing:")
print(f"Original: {df['text'][10]}")
print(f"Cleaned (Stemmed): {df['cleaned_text'][10]}")


Sample Preprocessing:
Original: Snacks were good but the legroom was terrible
Cleaned (Stemmed): snack good legroom terribl


In [92]:
#Step 4:
# a) Convert text to numerical vectors (TF-IDF)
# b) check X,y and shape len

vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df['cleaned_text']).toarray()
y = df['sentiment']

print(f"X (Feature Matrix) Shape: {X.shape}")
print(f"y (Target Vector) Shape: {y.shape}")
print(f"Number of unique features (words) in vectorizer: {len(vectorizer.get_feature_names_out())}")

X (Feature Matrix) Shape: (30, 97)
y (Target Vector) Shape: (30,)
Number of unique features (words) in vectorizer: 97


In [93]:
# Step 5: Split into train and test sets
# 80% training and 20% testing

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

Training samples: 24
Testing samples: 6


In [94]:
# Step 6: Train a Logistic Regression model
# a: Create Logistic Model
# b: Train Logistic Model

model = LogisticRegression()


model.fit(X_train, y_train)
print("Logistic Regression Model Trained.")

Logistic Regression Model Trained.


In [95]:
# Step 7:
# Evaluate accuracy and classification report
# a. predict Model
# b. Precision, recall, F1-Score for each sentiments

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy Score: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Accuracy Score: 0.1667

Classification Report:
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00         3
     neutral       0.00      0.00      0.00         1
    positive       0.20      0.50      0.29         2

    accuracy                           0.17         6
   macro avg       0.07      0.17      0.10         6
weighted avg       0.07      0.17      0.10         6



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [96]:
# Step 8: Predict sentiment for new example tweets

new_tweets = [
    "I had a fantastic flight with you! Everything was smooth and the service was excellent.",
    "My luggage is missing and your customer service is terrible. I will never fly this airline again.",
    "Flight 345 is delayed. Please provide an update on the gate change.",
    "Worst experience, ticket cancelled without notice. Avoid this airline."
]

# Preprocess and vectorize
cleaned_new_tweets = [preprocess_text_with_stemming(tweet) for tweet in new_tweets]
X_new = vectorizer.transform(cleaned_new_tweets)

# Predict
predictions = model.predict(X_new)


MAX_WIDTH = max(len(msg) for msg in new_tweets) + 3

for i, (msg, pred) in enumerate(zip(new_tweets, predictions)):
  print(f"{i+1}. {msg:<{MAX_WIDTH}} -->> {pred}")

1. I had a fantastic flight with you! Everything was smooth and the service was excellent.              -->> positive
2. My luggage is missing and your customer service is terrible. I will never fly this airline again.    -->> negative
3. Flight 345 is delayed. Please provide an update on the gate change.                                  -->> positive
4. Worst experience, ticket cancelled without notice. Avoid this airline.                               -->> negative
